# Import libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
sns.set_theme()

# encode
from sklearn.preprocessing import LabelEncoder

# machine learning
from sklearn.ensemble import RandomForestClassifier

# datetime
from datetime import datetime

# xử lý chuỗi
import re

# Setting to make numbers easier to read on display
pd.options.display.float_format = '{:20.2f}'.format

# Show all columns on output
pd.set_option('display.max_columns', 999)

In [ ]:
# connect to google drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


---

# Viewing data

In [ ]:
# load dataset
file_path = "/content/drive/MyDrive/0003. FTU SUBJECTS/010. PHÂN TÍCH DỮ LIỆU/04. CUSTOMER SEGMENTATION PROJECT/01. MATERIAL/02. DATASET GỐC/Raw_data.xlsx - Transactions.csv"
df = pd.read_csv(file_path)

In [ ]:
display(df.head())

,transaction_id,product_id,customer_id,transaction_date,online_order,order_status,brand,product_line,product_class,product_size,list_price,standard_cost,product_first_sold_date
0,1,2,2950,2/25/2017,False,Approved,Solex,Standard,medium,medium,"71,49","$53,62",41245.00
1,2,3,3120,5/21/2017,True,Approved,Trek Bicycles,Standard,medium,large,"2091,47","$388,92",41701.00
2,3,37,402,10/16/2017,False,Approved,OHM Cycles,Standard,low,medium,"1793,43","$248,82",36361.00
3,4,88,3135,8/31/2017,False,Approved,Norco Bicycles,Standard,medium,medium,"1198,46","$381,10",36145.00
4,5,78,787,10/1/2017,True,Approved,Giant Bicycles,Standard,medium,large,"1765,3","$709,48",42226.00


---

# Inspect Dataset

---

## Infomation

In [ ]:
# Hiển thị thông tin cơ bản về dữ liệu
print("Thông tin tổng quan về dataset:")
print(df.info())

Thông tin tổng quan về dataset:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 13 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   transaction_id           20000 non-null  int64  
 1   product_id               20000 non-null  int64  
 2   customer_id              20000 non-null  int64  
 3   transaction_date         20000 non-null  object 
 4   online_order             19640 non-null  object 
 5   order_status             20000 non-null  object 
 6   brand                    19803 non-null  object 
 7   product_line             19803 non-null  object 
 8   product_class            19803 non-null  object 
 9   product_size             19803 non-null  object 
 10  list_price               20000 non-null  object 
 11  standard_cost            19803 non-null  object 
 12  product_first_sold_date  19803 non-null  float64
dtypes: float64(1), int64(3), object(9)
memory us

---

### Mô tả ngắn gọn thông tin dữ liệu

Dữ liệu chứa thông tin về các giao dịch của một công ty bán xe đạp, với **20.000 bản ghi** và **13 cột**. Dưới đây là mô tả chi tiết từng cột:

| Tên cột                  | Mô tả                                      | Số giá trị không null | Kiểu dữ liệu |
|--------------------------|--------------------------------------------|-----------------------|--------------|
| `transaction_id`         | ID duy nhất cho mỗi giao dịch              | 20000                 | `int64`      |
| `product_id`             | ID sản phẩm được bán                       | 20000                 | `int64`      |
| `customer_id`            | ID khách hàng thực hiện giao dịch          | 20000                 | `int64`      |
| `transaction_date`       | Ngày diễn ra giao dịch                     | 20000                 | `object`     |
| `online_order`           | Giao dịch có được thực hiện trực tuyến hay không | 19640             | `object`     |
| `order_status`           | Trạng thái đơn hàng (ví dụ: "Approved")    | 20000                 | `object`     |
| `brand`                  | Thương hiệu sản phẩm                       | 19803                 | `object`     |
| `product_line`           | Dòng sản phẩm                              | 19803                 | `object`     |
| `product_class`          | Phân loại sản phẩm                         | 19803                 | `object`     |
| `product_size`           | Kích thước sản phẩm                        | 19803                 | `object`     |
| `list_price`             | Giá niêm yết của sản phẩm                  | 20000                 | `object`     |
| `standard_cost`          | Chi phí chuẩn của sản phẩm                 | 19803                 | `object`     |
| `product_first_sold_date`| Ngày sản phẩm được bán lần đầu tiên        | 19803                 | `float64`    |

---

## Tổng quan về kiểu dữ liệu

- **3 cột kiểu `int64`**: `transaction_id`, `product_id`, `customer_id`.
- **1 cột kiểu `float64`**: `product_first_sold_date`.
- **9 cột kiểu `object`**: Bao gồm các cột như `transaction_date`, `online_order`, `order_status`, `brand`, `product_line`, `product_class`, `product_size`, `list_price`, và `standard_cost`.

---

## Nhận xét sơ bộ về dữ liệu

1. **Các cột số nguyên (`int64`)**:
   - Các cột này đại diện cho các ID duy nhất (`transaction_id`, `product_id`, `customer_id`) và không có giá trị bị thiếu.

2. **Các cột kiểu `object`**:
   - Một số cột như `online_order`, `brand`, `product_line`, `product_class`, `product_size`, và `standard_cost` có giá trị bị thiếu (khoảng **197 giá trị**).
   - Cột `list_price` và `standard_cost` đang ở dạng chuỗi (`object`) và cần được chuyển đổi sang kiểu số để phân tích.
   - Cột `transaction_date` cần được chuyển đổi sang định dạng ngày tháng (`datetime`).

3. **Cột kiểu `float64`**:
   - Cột `product_first_sold_date` đang ở định dạng số thực (`float64`) nhưng cần được chuyển đổi thành định dạng ngày tháng (`datetime`).

4. **Phát hiện bất thường**:
   - Một số cột có giá trị bị thiếu cần được xử lý (xóa hoặc điền giá trị hợp lý).
   - Cần kiểm tra tính nhất quán của dữ liệu trong các cột như `brand`, `product_line`, và `product_class`.
   - Không phát hiện bản ghi trùng lặp trong dataset.

---

## Kết luận

Dataset này cung cấp thông tin chi tiết về các giao dịch, sản phẩm, và khách hàng. Tuy nhiên, cần thực hiện các bước làm sạch dữ liệu như xử lý giá trị bị thiếu, chuyển đổi kiểu dữ liệu, và kiểm tra tính nhất quán trước khi tiến hành phân tích sâu hơn.

---

## Describe

In [ ]:
# Numerical
display(df.describe().T)

,count,mean,std,min,25%,50%,75%,max
transaction_id,20000.00,10000.50,5773.65,1.00,5000.75,10000.50,15000.25,20000.00
product_id,20000.00,45.36,30.75,0.00,18.00,44.00,72.00,100.00
customer_id,20000.00,1738.25,1011.95,1.00,857.75,1736.00,2613.00,5034.00
product_first_sold_date,19803.00,38199.78,2875.20,33259.00,35667.00,38216.00,40672.00,42710.00


---
---

### Nhận xét chung dữ liệu số

## Tổng quan chung:

### **Chất lượng dữ liệu**:
- Dữ liệu nhìn chung sạch sẽ, không có giá trị bất thường rõ ràng ngoại trừ một số trường hợp cần kiểm tra thêm (ví dụ: `product_id = 0`, giá trị bị thiếu trong `product_first_sold_date`).
- Tất cả các cột đều có phân bố hợp lý, không có sự chênh lệch quá lớn giữa các giá trị **min**, **max**, và **median**.

---

### **Đặc điểm nổi bật**:
- Số lượng giao dịch (**20.000**) vượt xa số lượng khách hàng (**5034**), cho thấy mỗi khách hàng có thể thực hiện nhiều giao dịch.
- Ngày sản phẩm được bán lần đầu tiên (`product_first_sold_date`) có một số giá trị bị thiếu, cần xử lý trước khi phân tích sâu hơn.

---

### **Đề xuất tiếp theo**:
1. **Xử lý giá trị bị thiếu** trong cột `product_first_sold_date`.
2. **Kiểm tra lại giá trị `product_id = 0`** để xác định xem có phải lỗi nhập liệu hay không.
3. **Chuyển đổi cột `product_first_sold_date`** sang định dạng ngày tháng dễ đọc hơn (nếu cần).

---

In [ ]:
# Object
display(df.describe(include='object').T)

,count,unique,top,freq
transaction_date,20000,364,2/14/2017,82
online_order,19640,2,True,9829
order_status,20000,2,Approved,19821
brand,19803,6,Solex,4253
product_line,19803,4,Standard,14176
product_class,19803,3,medium,13826
product_size,19803,3,medium,12990
list_price,20000,296,"2091,47",465
standard_cost,19803,103,"$388,92",465


---

###Nhận xét chung dữ liệu chữ

## **Kết luận chung**

### **Chất lượng dữ liệu**:
- Dữ liệu nhìn chung sạch sẽ, nhưng có một số cột chứa giá trị bị thiếu (chủ yếu là **197 giá trị**, chiếm khoảng **1%**). Những giá trị này cần được xử lý trước khi phân tích sâu hơn.
- Định dạng dữ liệu cần được chuẩn hóa (ví dụ: ngày tháng, dấu phẩy trong giá tiền).

---

### **Đặc điểm nổi bật**:
- Ngày **14/2/2017** có nhiều giao dịch nhất, có thể do các chiến dịch marketing hoặc sự kiện đặc biệt.
- Thương hiệu **Solex** và dòng sản phẩm **Standard** chiếm đa số, cho thấy sự ưa chuộng của khách hàng.
- Phân loại sản phẩm và kích thước phổ biến nhất là **medium**, cần tối ưu hóa danh mục sản phẩm xung quanh phân khúc này.

---

### **Đề xuất tiếp theo**:
1. Xử lý giá trị bị thiếu bằng cách điền giá trị hợp lý hoặc loại bỏ nếu không ảnh hưởng.
2. Chuẩn hóa định dạng dữ liệu (ngày tháng, giá tiền).
3. Phân tích sâu hơn về mối quan hệ giữa các cột để tìm ra xu hướng và cơ hội kinh doanh tiềm năng.

---

# Data Cleaning

## Tạo dataframe copy

In [ ]:
df_clean = df.copy()

## Xử lý định dạng dữ liệu

### Chuyển đổi cột list_price và standard_cost sang kiểu số
- Loại bỏ ký tự đặc biệt (dấu phẩy, ký hiệu $) và chuyển đổi sang kiểu số.

In [ ]:
# Chuyển sang kiểu chuỗi 100%
df_clean['list_price'] = df_clean['list_price'].astype(str)
df_clean['standard_cost'] = df_clean['standard_cost'].astype(str)

# loại bỏ khoảng trắng thừa
df_clean['list_price'] = df_clean['list_price'].str.strip()
df_clean['standard_cost'] = df_clean['standard_cost'].str.strip()

In [ ]:
# Xử lý cột list_price
df_clean['list_price'] = df_clean['list_price'].str.replace(',', '.').astype(float)

In [ ]:
# Hàm xử lý cột standard_cost
def clean_standard_cost(value):
    if isinstance(value, str):  # Kiểm tra nếu giá trị là chuỗi
        # Bước 1: Loại bỏ ký tự '$'
        value = value.replace('$', '')

        # Bước 2: Thay thế dấu ',' bằng dấu '.' để chuẩn hóa thập phân
        value = value.replace(',', '.')

        # Bước 3: Xử lý dấu chấm cho hàng nghìn (ví dụ: '1.479.11' -> '1479.11')
        parts = value.split('.')
        if len(parts) > 2:  # Nếu có nhiều hơn 2 phần, có dấu chấm cho hàng nghìn
            value = ''.join(parts[:-1]) + '.' + parts[-1]  # Ghép các phần trước và thêm dấu thập phân

        # Bước 4: Chuyển đổi sang float
        return float(value)
    return value  # Trả về giá trị gốc nếu không phải chuỗi

# Áp dụng hàm vào cột standard_cost
df_clean['standard_cost'] = df_clean['standard_cost'].apply(clean_standard_cost)

In [ ]:
df_clean[['list_price', 'standard_cost']].describe()

,list_price,standard_cost
count,20000.00,19803.00
mean,1107.83,556.05
std,582.83,405.96
min,12.01,7.21
25%,575.27,215.14
50%,1163.89,507.58
75%,1635.30,795.10
max,2091.47,1759.85


---

### Chuyển đổi cột transaction_date sang định dạng ngày tháng (datetime)

In [ ]:
# Chuyển đổi cột transaction_date sang datetime
df_clean['transaction_date'] = pd.to_datetime(df_clean['transaction_date'], format='%m/%d/%Y')

In [ ]:
# Kiểm tra type của transaction_date
df_clean['transaction_date'].dtypes

dtype('<M8[ns]')

### Chuyển đổi cột product_first_sold_date sang định dạng ngày tháng (datetime)
- Cột này hiện đang ở dạng số nguyên (số ngày từ mốc thời gian cố định).

In [ ]:
# Chuyển đổi từ số ngày sang datetime
df_clean['product_first_sold_date'] = pd.to_datetime('1899-12-30') + pd.to_timedelta(df_clean['product_first_sold_date'], unit='D')

In [ ]:
df_clean['product_first_sold_date'].head()

,product_first_sold_date
0,2012-12-02
1,2014-03-03
2,1999-07-20
3,1998-12-16
4,2015-08-10


## Xử lý giá trị bị thiếu

In [ ]:
# check missing value
missing = df_clean.isna().sum().sort_values(ascending=False)
missing = missing[missing > 0]
missing

,0
online_order,360
brand,197
product_line,197
product_class,197
product_size,197
standard_cost,197
product_first_sold_date,197


In [ ]:
# percentage
missing_percent = (missing / df_clean.shape[0]) * 100
missing_percent

,0
online_order,1.80
brand,0.98
product_line,0.98
product_class,0.98
product_size,0.98
standard_cost,0.98
product_first_sold_date,0.98


### Đối với cột product_first_sold_date

In [ ]:
# Điền giá trị trung vị cho cột product_first_sold_date
df_clean['product_first_sold_date'].fillna(df_clean['product_first_sold_date'].median(), inplace=True)

<ipython-input-20-2eb6808ac5c5>:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_clean['product_first_sold_date'].fillna(df_clean['product_first_sold_date'].median(), inplace=True)


In [ ]:
# check missing value
df_clean['product_first_sold_date'].isna().sum()

0

---

### Đối với các cột khác có giá trị bị thiếu

In [ ]:
missing_columns = ['online_order', 'brand', 'product_line', 'product_class', 'product_size', 'standard_cost']

for column in missing_columns:
    df_clean[column].fillna(df_clean[column].mode()[0], inplace=True)

<ipython-input-22-2f59eac2f405>:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_clean[column].fillna(df_clean[column].mode()[0], inplace=True)
<ipython-input-22-2f59eac2f405>:4: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_clean[column].fillna(df_clean[column].mode()[0], inplace=True)


In [ ]:
# check missing value
missing = df_clean.isna().sum().sort_values(ascending=False)
missing = missing[missing > 0]
missing

,0


---

## Kiểm tra và xử lý dữ liệu bất thường

### Kiểm tra giá trị product_id = 0
- Kiểm tra tần suất xuất hiện của product_id = 0.

In [ ]:
# Kiểm tra tần suất của product_id = 0
print(df['product_id'].value_counts())

product_id
0      1378
3       354
1       311
35      268
38      267
       ... 
71      137
8       136
16      136
100     130
47      121
Name: count, Length: 101, dtype: int64


---

### Kiểm tra dữ liệu trùng lặp

In [ ]:
# Kiểm tra dữ liệu trùng lặp
duplicate_rows = df_clean.duplicated()
print("\nSố lượng bản ghi trùng lặp:", duplicate_rows.sum())

# Nếu có bản ghi trùng lặp, loại bỏ chúng
if duplicate_rows.sum() > 0:
    data_cleaned = df_clean.drop_duplicates()
    print("Đã loại bỏ các bản ghi trùng lặp.")


Số lượng bản ghi trùng lặp: 0


---

## Tạo cột Profit (lợi nhuận)

In [ ]:
# Tạo cột Profit dựa trên công thức: Profit = List Price - Standard Cost
df_clean['profit'] = df_clean['list_price'] - df_clean['standard_cost']

## So sánh data clean và data gốc

In [ ]:
percentage = len(df_clean) / len(df) * 100
print(f'Data clean chiếm  {percentage:.2f}% so với data gốc')

Data clean chiếm  100.00% so với data gốc


## Sắp xếp lại các cột

In [ ]:
print(f'Số cột của data gốc: {len(df.columns)}')
print(f'Số cột của data clean: {len(df_clean.columns)}')

Số cột của data gốc: 13
Số cột của data clean: 14


In [ ]:
# chuyển online_order thành kiểu object
df_clean['online_order'] = df_clean['online_order'].astype(str)

In [ ]:
# Lấy danh sách các cột kiểu object
object_columns = df_clean.select_dtypes(include=['object']).columns.tolist()

# Lấy danh sách các cột kiểu numeric
numeric_columns = df_clean.select_dtypes(include=['int64', 'float64']).columns.tolist()

# lấy danh sách cột kiểu datetime
datetime_columns = ['transaction_date', 'product_first_sold_date']

# Sắp xếp lại thứ tự cột: Numeric -> Object
df_clean = df_clean[numeric_columns + object_columns + datetime_columns]

# Kiểm tra số cột
print(f'Số cột của data clean sau khi sắp xếp cột : {len(df_clean.columns)}')

Số cột của data clean sau khi sắp xếp cột : 14


In [ ]:
df_clean.head()

,transaction_id,product_id,customer_id,list_price,standard_cost,profit,online_order,order_status,brand,product_line,product_class,product_size,transaction_date,product_first_sold_date
0,1,2,2950,71.49,53.62,17.87,False,Approved,Solex,Standard,medium,medium,2017-02-25,2012-12-02
1,2,3,3120,2091.47,388.92,1702.55,True,Approved,Trek Bicycles,Standard,medium,large,2017-05-21,2014-03-03
2,3,37,402,1793.43,248.82,1544.61,False,Approved,OHM Cycles,Standard,low,medium,2017-10-16,1999-07-20
3,4,88,3135,1198.46,381.10,817.36,False,Approved,Norco Bicycles,Standard,medium,medium,2017-08-31,1998-12-16
4,5,78,787,1765.30,709.48,1055.82,True,Approved,Giant Bicycles,Standard,medium,large,2017-10-01,2015-08-10


----

## Lưu data đã được làm sạch